# Day S1 — 서열 데이터베이스와 레코드 읽기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/supplement/day-S1.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/supplement/day-S1.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — FASTA 읽기

In [ ]:
fasta_text = """>demo|PRT001 가상 단백질 1 (연습용)
MKTEGVLAAQ
EKTKQGVAEA
>demo|PRT002 가상 단백질 2 (연습용)
MKSEGVVAAA
EKTK
"""

def read_fasta(text):
    """FASTA 글자를 {헤더: 서열} 딕셔너리로 바꾼다."""
    records = {}
    header = None
    for line in text.splitlines():
        line = line.strip()
        if line == "":
            continue
        if line.startswith(">"):
            header = line[1:]          # '>' 를 떼고 헤더로 쓴다
            records[header] = ""
        else:
            records[header] += line    # 여러 줄에 나뉜 서열을 이어 붙인다
    return records

records = read_fasta(fasta_text)
for header, seq in records.items():
    print(header.split()[0], len(seq), seq)

### 예시 2 — GenBank 형식 레코드에서 CDS 꺼내기

In [ ]:
record_text = """LOCUS       DEMO0001                  45 bp    mRNA    linear   SYN 01-JAN-2000
DEFINITION  Synthetic practice record (not a real sequence).
ACCESSION   DEMO0001
VERSION     DEMO0001.2
FEATURES             Location/Qualifiers
     source          1..45
                     /organism="synthetic construct"
     CDS             7..36
                     /gene="demoA"
                     /product="practice peptide"
ORIGIN
        1 ggcaccatgg ctgaaaaagg tgttctggca gcataaccgt tcaaa
//
"""

info = {}
sequence = ""
in_origin = False
for line in record_text.splitlines():
    if line.startswith("//"):
        break
    if in_origin:
        parts = line.split()
        sequence += "".join(parts[1:])        # 맨 앞 숫자는 버리고 염기만 모은다
    elif line.startswith("ORIGIN"):
        in_origin = True
    elif line.startswith(("LOCUS", "DEFINITION", "ACCESSION", "VERSION")):
        key, value = line.split(maxsplit=1)
        info[key] = value
    elif line.strip().startswith("CDS"):
        info["CDS"] = line.split()[1]

sequence = sequence.upper()
start, end = info["CDS"].split("..")
start, end = int(start), int(end)
cds = sequence[start - 1:end]                 # GenBank은 1부터, Python은 0부터 센다

print("accession:", info["ACCESSION"])
print("version  :", info["VERSION"])
print("전체 길이:", len(sequence))
print("CDS 위치 :", info["CDS"], "→ 길이", len(cds))
print("CDS 서열 :", cds)

### 예시 3 — 여섯 읽기 틀로 번역하기

In [ ]:
bases = "TCAG"
amino = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
codon_table = {}
i = 0
for b1 in bases:
    for b2 in bases:
        for b3 in bases:
            codon_table[b1 + b2 + b3] = amino[i]
            i += 1

def translate(dna):
    protein = ""
    for pos in range(0, len(dna) - 2, 3):
        protein += codon_table[dna[pos:pos + 3]]
    return protein

def reverse_complement(dna):
    pair = {"A": "T", "T": "A", "G": "C", "C": "G"}
    return "".join(pair[base] for base in reversed(dna))

assert len(codon_table) == 64
assert translate("ATGGCTTAA") == "MA*"
assert reverse_complement("ATGC") == "GCAT"

dna = "GGCACCATGGCTGAAAAAGGTGTTCTGGCAGCATAACCGTTCAAA"
for strand_name, strand in [("+", dna), ("-", reverse_complement(dna))]:
    for frame in range(3):
        print(f"{strand_name}{frame + 1}: {translate(strand[frame:])}")

## 직접 해보기

5. **(선택) 실제 FASTA를 내 함수로 읽기.** UniProt 항목의 Download에서 FASTA를 받아 프로젝트의 `data/` 폴더에 저장한 뒤, 파일 내용을 읽어 예시 1의 `read_fasta`에 넣습니다.

In [ ]:
with open("data/P37840.fasta") as handle:
    records = read_fasta(handle.read())
for header, seq in records.items():
    print(header[:40], len(seq))